## Extracting the complexity measures 

In [1]:
import os, random, tqdm, torch, warnings

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from multiprocessing import Pool
from pymfe.mfe import MFE
from tabicl_prior import sample_datasets

In [4]:
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [5]:
N_DATASETS = 10
SEED = 42
N_JOBS = 4

In [6]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [7]:
device = 'cuda' if torch.accelerator.is_available() else 'cpu'
device

'cuda'

## Real datasets

In [8]:
def normalize(file):
    df = pd.read_csv(file, na_values=['?', 'Not_Say'], low_memory=False)
    X, y = df.iloc[:, :-1], df.iloc[:, -1]

    X = X.dropna(axis=1, how='all')
    cat = X.select_dtypes(exclude='number').columns
    num = X.columns.difference(cat)

    X[num] = X[num].fillna(X[num].mean())
    X[cat] = X[cat].fillna('missing')

    return X.to_numpy(), y.to_numpy()

In [9]:
def load_datasets(path):
    return [normalize(f'{path}/{file}') for file in os.listdir(path)]

In [10]:
def extract(args):
    i, X, y = args
    mfe = MFE(groups='complexity', features='all', summary='mean', random_state=SEED)
    mfe.fit(X, y, suppress_warnings=True)
    measures, values = mfe.extract(suppress_warnings=True)
    return {'id': i, 'n_classes': len(np.unique(y)), 'n_rows': X.shape[0], 'n_cols': X.shape[1], **dict(zip(measures, map(float, values)))}

In [11]:
real = load_datasets('tabarena/') + load_datasets('talent/')
len(real)

238

In [12]:
data_tuples = [(i, X, y) for i, (X, y) in enumerate(real)]
data_tuples = data_tuples[:4]
len(data_tuples)

4

In [13]:
with Pool(N_JOBS) as pool:   
    complexity_real = list(tqdm.tqdm(pool.imap(extract, data_tuples), total=len(data_tuples)))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [03:24<00:00, 51.07s/it]


In [14]:
df = pd.DataFrame(complexity_real)
df

,id,n_classes,n_rows,n_cols,c1,c2,cls_coef,density,f1.mean,f1v.mean,...,l3.mean,lsc,n1,n2.mean,n3.mean,n4.mean,t1,t2,t3,t4
0,0,3,1014,6,0.988027,0.019554,0.357968,0.907676,0.857807,NaN,...,0.185338,0.992288,0.457594,NaN,0.160750,0.526627,0.183432,0.005917,0.003945,0.666667
1,1,2,2240,25,0.607601,0.659969,0.520869,0.839316,0.998501,NaN,...,0.091071,0.992611,0.257143,NaN,0.157589,0.191071,0.589286,0.310714,0.000446,0.001437
2,2,2,2400,30,1.000000,0.000000,0.287237,0.881858,0.916966,NaN,...,0.145000,0.993565,0.223750,0.383195,0.094167,0.432917,0.565833,0.012500,0.003750,0.300000
3,3,2,1000,20,0.881291,0.275862,0.593987,0.875692,0.989011,NaN,...,0.194000,0.994182,0.432000,0.443736,0.288000,0.262000,0.702000,0.049000,0.001000,0.020408


In [15]:
df.to_csv('complexity_real.csv')

## Synthetic datasets

In [16]:
synthetic = sample_datasets('clf', stage=1, n_datasets=N_DATASETS, seed=SEED, n_jobs=N_JOBS)

In [17]:
complexity_synthetic = []
for i, data in enumerate(synthetic):
    X = np.concatenate([data['X_train'], data['X_test']], axis=0)
    y = np.concatenate([data['y_train'], data['y_test']], axis=0)
    complexity_synthetic.append(extract((i, X, y)))   

In [18]:
df = pd.DataFrame(complexity_synthetic)
df

,id,n_classes,n_rows,n_cols,c1,c2,cls_coef,density,f1.mean,f1v.mean,...,l3.mean,lsc,n1,n2.mean,n3.mean,n4.mean,t1,t2,t3,t4
0,0,2,1024,45,0.518370,0.741486,0.584885,0.810969,0.552055,NaN,...,0.000000,0.205412,0.001953,0.012261,0.000000,0.156250,0.001953,0.043945,0.018555,0.422222
1,1,4,1024,68,0.758671,0.320394,0.502359,0.914687,0.989454,NaN,...,0.219192,0.997641,0.652344,0.501483,0.509766,0.372070,0.995117,0.066406,0.038086,0.573529
2,2,2,1024,66,0.666387,0.597035,0.615591,0.861613,0.986429,NaN,...,0.029297,0.988946,0.254883,0.447899,0.166016,0.214844,0.866211,0.064453,0.050781,0.787879
3,3,8,1024,68,0.952895,0.033776,0.462253,0.958900,0.960843,NaN,...,0.129415,0.995941,0.669922,0.490054,0.493164,0.407227,0.965820,0.066406,0.054688,0.823529
4,4,9,1024,44,0.926380,0.043525,0.135104,0.948829,0.754535,NaN,...,0.065617,0.987629,0.204102,0.159395,0.049805,0.610352,0.299805,0.042969,0.013672,0.318182
5,5,8,1024,44,0.824195,0.215730,0.141865,0.906195,0.683203,NaN,...,0.018055,0.958012,0.156250,NaN,0.048828,0.529297,0.150391,0.042969,0.025391,0.590909
6,6,2,1024,44,0.936315,0.160038,0.622204,0.890887,0.989266,NaN,...,0.146484,0.997204,0.551758,0.481476,0.367188,0.239258,0.937500,0.042969,0.035156,0.818182
7,7,5,1024,44,0.771566,0.228623,0.307101,0.900954,0.890006,NaN,...,0.083175,0.977399,0.350586,0.393133,0.221680,0.506836,0.554688,0.042969,0.019531,0.454545
8,8,3,1024,1,0.789170,0.190346,0.207348,0.911978,0.792324,NaN,...,0.163699,0.997694,0.702148,0.482670,0.472656,0.509766,0.579102,0.000977,0.000977,1.000000
9,9,6,1024,1,0.975591,0.018298,1.000000,1.000000,0.385230,NaN,...,0.219461,1.000000,0.947266,NaN,0.713867,0.744141,0.001953,0.000977,0.000977,1.000000


In [19]:
df.to_csv('complexity_sintetic.csv')